In [0]:
%pip install pmdarima statsmodels

In [0]:
import mlflow
import mlflow.pyfunc
import numpy as np
import pandas as pd
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import pickle, os

# Cell 1: Load and split
pdf = spark.table("weather_silver").toPandas().sort_values("date").set_index("date")
ts = pdf["avg_temp"].dropna()

train_size = int(len(ts) * 0.8)
train, test = ts[:train_size], ts[train_size:]



In [0]:
# Cell 2: auto_arima to find best order
print("Running auto_arima — this takes ~2-3 minutes...")
auto_model = auto_arima(
    train,
    seasonal=True, m=7,        # m=7 → weekly seasonality on daily data
    d=1, D=1,
    max_p=3, max_q=3,
    max_P=2, max_Q=2,
    information_criterion="aic",
    trace=True,
    error_action="ignore",
    suppress_warnings=True
)
print(f"Best order: {auto_model.order}  Seasonal order: {auto_model.seasonal_order}")
print(f"Best AIC: {auto_model.aic():.2f}")



In [0]:
# Cell 3: MLflow experiment — log everything
mlflow.set_experiment("/Users/santhoshnagendrarajan@gmail.com/weather-sarima")

def evaluate_and_log(order, seasonal_order, run_name):
    with mlflow.start_run(run_name=run_name):
        model = SARIMAX(train, order=order, seasonal_order=seasonal_order,
                        enforce_stationarity=False, enforce_invertibility=False)
        result = model.fit(disp=False)

        forecast = result.forecast(steps=len(test))
        rmse = np.sqrt(mean_squared_error(test, forecast))
        mae  = mean_absolute_error(test, forecast)
        mape = np.mean(np.abs((test.values - forecast.values) / test.values)) * 100

        # Log params
        mlflow.log_param("p", order[0]); mlflow.log_param("d", order[1]); mlflow.log_param("q", order[2])
        mlflow.log_param("P", seasonal_order[0]); mlflow.log_param("S", seasonal_order[3])
        mlflow.log_param("aic", round(result.aic, 2))

        # Log metrics
        mlflow.log_metric("rmse", round(rmse, 4))
        mlflow.log_metric("mae",  round(mae, 4))
        mlflow.log_metric("mape", round(mape, 4))

        # Save forecast plot as artifact
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(train[-60:], label="Train (last 60 days)", color="#1D9E75")
        ax.plot(test, label="Actual", color="#378ADD")
        ax.plot(test.index, forecast, label="Forecast", color="#D85A30", linestyle="--")
        ax.fill_between(test.index,
                        result.get_forecast(steps=len(test)).conf_int().iloc[:,0],
                        result.get_forecast(steps=len(test)).conf_int().iloc[:,1],
                        alpha=0.2, color="#D85A30")
        ax.legend(); ax.set_title(f"{run_name} | RMSE={rmse:.2f}")
        fig.savefig("/tmp/forecast_plot.png", dpi=150)
        mlflow.log_artifact("/tmp/forecast_plot.png")

        # Save model pickle as artifact
        with open("/tmp/sarima_model.pkl", "wb") as f:
            pickle.dump(result, f)
        mlflow.log_artifact("/tmp/sarima_model.pkl")

        print(f"{run_name} → RMSE: {rmse:.3f}  MAE: {mae:.3f}  MAPE: {mape:.2f}%")
        return rmse, result


In [0]:
# Cell 4: Run 3 experiments to compare
best_order = auto_model.order
best_seasonal = auto_model.seasonal_order

evaluate_and_log(best_order,       best_seasonal,          "auto_arima_best")
evaluate_and_log((1, 1, 1),        (1, 1, 0, 7),           "baseline_simple")
evaluate_and_log((best_order[0],1,best_order[2]+1), best_seasonal, "tuned_q_plus1")